# n=8 Retraining + Faster Coupling Search



This notebook explores the first recommendation from the algebraic reconstruction notebook: retrain the graph-recovery pipeline in the regime where the exact algebraic search is still fully available.



The key change is to work with **8-vertex graphs while keeping one distinguished vertex**. That leaves a 7-vertex complement, and the NetworkX graph atlas is still complete at that size. So for each target reduction we can recover the full exact family of admissible 8-vertex unfoldings instead of only partial information.



We also implement a faster coupling search. The acceleration is simple but effective:



1. Filter complement graphs $H$ by a necessary denominator divisibility condition.

2. For the surviving graphs, screen binary coupling vectors $c \in \{0,1\}^7$ numerically at a few safe $\lambda$ values.

3. Only run the expensive symbolic check on the tiny shortlist that survives both filters.



The notebook then uses these exact families to build a small rooted multi-label training set and retrains a compact MLP on 8-vertex examples only.

In [1]:
import itertools
import time

import numpy as np
import networkx as nx
import sympy as sp
import torch
import torch.nn as nn
import torch.optim as optim

lam = sp.symbols('lambda')
device = 'cpu'
np.random.seed(0)
torch.manual_seed(0)

print('numpy', np.__version__)
print('networkx', nx.__version__)
print('sympy', sp.__version__)
print('torch', torch.__version__)

numpy 2.3.3
networkx 3.5
sympy 1.14.0
torch 2.8.0+cu128


## Exact-Size Setup

We work in the **rooted** setting: vertex `0` is always the kept vertex, and vertices `1..7` form the complement. This matters for learning. The reduction is computed relative to a distinguished root, so we should not quotient out by arbitrary full-graph isomorphism. Instead, we deduplicate by isomorphisms that preserve the root and only allow relabellings of the complement.

Because $|S| = 1$, the reduction feature is a scalar rational function sampled across a $\lambda$ grid. That keeps the ML side lightweight: the input feature is just a vector of sampled reduction values, while the target remains the upper-triangle edge vector of an 8-vertex graph.

In [2]:
def reduced_expr(expr):
    return sp.cancel(sp.together(sp.simplify(expr)))

def compute_reduction_symbolic_from_graph(G, kept=0):
    A = sp.Matrix(nx.to_numpy_array(G, dtype=int).tolist())
    sbar = [i for i in range(G.number_of_nodes()) if i != kept]
    c = sp.Matrix([A[i, kept] for i in sbar])
    H = sp.Matrix([[A[i, j] for j in sbar] for i in sbar])
    return reduced_expr(-(c.T * (H - lam * sp.eye(len(sbar))).inv() * c)[0, 0])

def isospectral_reduction_numeric(A, kept, lambdas):
    n = A.shape[0]
    sbar = [i for i in range(n) if i != kept]
    c = A[np.ix_(sbar, [kept])]
    H = A[np.ix_(sbar, sbar)]
    out = []
    for lam_val in lambdas:
        M = H - lam_val * np.eye(len(sbar))
        try:
            inv = np.linalg.inv(M)
        except np.linalg.LinAlgError:
            inv = np.linalg.pinv(M + 1e-8 * np.eye(len(sbar)))
        out.append(float(-(c.T @ inv @ c)[0, 0]))
    return np.array(out, dtype=np.float32)

def assemble_full_graph(H, c_tuple):
    k = H.number_of_nodes()
    A = np.zeros((k + 1, k + 1), dtype=int)
    for i, ci in enumerate(c_tuple):
        A[0, i + 1] = ci
        A[i + 1, 0] = ci
    A[1:, 1:] = nx.to_numpy_array(H, dtype=int)
    return A

def rooted_isomorphic(A, B):
    G1 = nx.from_numpy_array(A)
    G2 = nx.from_numpy_array(B)
    for G in (G1, G2):
        for node in G.nodes:
            G.nodes[node]['root'] = int(node == 0)
    node_match = nx.algorithms.isomorphism.categorical_node_match('root', 0)
    return nx.is_isomorphic(G1, G2, node_match=node_match)

def dedupe_rooted(adjacencies):
    reps = []
    for A in adjacencies:
        if not any(rooted_isomorphic(A, rep) for rep in reps):
            reps.append(A)
    return reps

def edge_vector(A):
    iu = np.triu_indices(A.shape[0], k=1)
    return A[iu].astype(np.float32)

def permute_complement(A, perm):
    order = np.array([0] + [p + 1 for p in perm], dtype=int)
    return A[np.ix_(order, order)]

def graph_edges(A):
    return sorted(nx.from_numpy_array(A).edges())

## Faster Coupling Search

The naive exact search would inspect all 1044 unlabeled 7-vertex complement graphs and all $2^7 - 1 = 127$ nonzero couplings for each one, for a total of 132,588 candidate pairs before symbolic verification. That is still manageable in principle, but wasteful.

The filter we use here is:

- **Graph-level spectral filter:** if the reduced denominator of the target rational function does not divide $\det(A_H - \lambda I)$, then $H$ cannot possibly produce the target reduction.
- **Coupling-level numeric screen:** after a graph survives, test each binary coupling on a few safe $\lambda$ values far from the poles. Only if the numeric values agree with the target do we pay for a symbolic check.

The one-time expensive step is precomputing the 7-vertex atlas cache. After that, each new target reduction is very fast to screen exactly.

In [3]:
def build_atlas_cache(k=7):
    cache = []
    t0 = time.time()
    for G0 in nx.graph_atlas_g():
        if G0.number_of_nodes() != k or nx.number_of_selfloops(G0) > 0:
            continue
        H = nx.convert_node_labels_to_integers(G0)
        A_np = nx.to_numpy_array(H, dtype=float)
        A_sp = sp.Matrix(A_np.astype(int).tolist())
        charpoly = sp.Poly(sp.expand((A_sp - lam * sp.eye(k)).det()), lam)
        eigs = np.linalg.eigvalsh(A_np)
        cache.append({
            'H': H,
            'A_np': A_np,
            'A_sp': A_sp,
            'charpoly': charpoly,
            'eigs': eigs,
        })
    return cache, time.time() - t0

def exact_size_family(target_r, cache, k=7, sample_points=None, numeric_tol=1e-7):
    if sample_points is None:
        sample_points = [-9.0, -7.0, -5.0, 5.0, 7.0, 9.0]

    target = reduced_expr(target_r)
    _, den = sp.fraction(target)
    den_poly = sp.Poly(sp.expand(den), lam)
    target_fn = sp.lambdify(lam, target, 'numpy')

    stats = {
        'atlas_graphs': len(cache),
        'graphs_after_denominator_filter': 0,
        'couplings_total': len(cache) * ((2 ** k) - 1),
        'couplings_after_numeric_filter': 0,
        'symbolic_checks': 0,
        'matches_before_dedupe': 0,
    }

    candidates = []
    for item in cache:
        _, rem = sp.div(item['charpoly'], den_poly)
        if rem != 0:
            continue
        stats['graphs_after_denominator_filter'] += 1

        usable = [x for x in sample_points if np.min(np.abs(item['eigs'] - x)) > 1e-8]
        if len(usable) < 4:
            continue
        target_vals = np.array([complex(target_fn(x)) for x in usable[:4]])
        M_inv = None

        for bits in itertools.product([0, 1], repeat=k):
            if not any(bits):
                continue
            c = np.array(bits, dtype=float)
            numeric_ok = True
            for x, target_val in zip(usable[:4], target_vals):
                val = -(c @ np.linalg.inv(item['A_np'] - x * np.eye(k)) @ c)
                if abs(val - target_val) > numeric_tol:
                    numeric_ok = False
                    break
            if not numeric_ok:
                continue

            stats['couplings_after_numeric_filter'] += 1
            if M_inv is None:
                M_inv = (item['A_sp'] - lam * sp.eye(k)).inv()

            stats['symbolic_checks'] += 1
            c_sp = sp.Matrix(list(bits))
            expr = reduced_expr(-(c_sp.T * M_inv * c_sp)[0, 0])
            if sp.simplify(expr - target) == 0:
                A_full = assemble_full_graph(item['H'], bits)
                if nx.is_connected(nx.from_numpy_array(A_full)):
                    stats['matches_before_dedupe'] += 1
                    candidates.append(A_full)

    family = dedupe_rooted(candidates)
    stats['matches_after_dedupe'] = len(family)
    return family, stats

def find_connected_graph_with_family_size(cache, min_family_size=2, p=0.35, seed=2026, max_trials=100):
    rng = np.random.default_rng(seed)
    fallback = None
    for trial in range(max_trials):
        G = nx.erdos_renyi_graph(8, p, seed=int(rng.integers(0, 10**9)))
        if not nx.is_connected(G):
            continue
        G = nx.convert_node_labels_to_integers(G)
        target_r = compute_reduction_symbolic_from_graph(G, kept=0)
        family, stats = exact_size_family(target_r, cache, k=7)
        if fallback is None and family:
            fallback = (G, target_r, family, stats, trial + 1)
        if len(family) >= min_family_size:
            return G, target_r, family, stats, trial + 1
    if fallback is None:
        raise RuntimeError('No connected graph with a nonempty exact-size family was found.')
    return fallback

In [4]:
atlas_cache, cache_build_seconds = build_atlas_cache(k=7)
print(f'Built 7-vertex atlas cache with {len(atlas_cache)} graphs in {cache_build_seconds:.2f}s')

test_graph, test_target_r, test_family, test_search_stats, test_trials = find_connected_graph_with_family_size(
    atlas_cache,
    min_family_size=2,
    p=0.35,
    seed=2026,
    max_trials=100,
)
test_A = nx.to_numpy_array(test_graph, dtype=int)

print('Held-out 8-vertex test graph found after', test_trials, 'connected trials')
print('Test graph edges:', graph_edges(test_A))
print('Family size:', len(test_family))
print('Search stats:', test_search_stats)
for idx, A_family in enumerate(test_family, start=1):
    print(f'  Family member {idx}: edges={graph_edges(A_family)} rooted_iso_to_source={rooted_isomorphic(A_family, test_A)}')

Built 7-vertex atlas cache with 1044 graphs in 152.93s
Held-out 8-vertex test graph found after 3 connected trials
Test graph edges: [(0, 3), (0, 5), (0, 6), (0, 7), (1, 2), (1, 3), (1, 6), (2, 4), (3, 5), (3, 6), (4, 5), (4, 7)]
Family size: 2
Search stats: {'atlas_graphs': 1044, 'graphs_after_denominator_filter': 2, 'couplings_total': 132588, 'couplings_after_numeric_filter': 4, 'symbolic_checks': 4, 'matches_before_dedupe': 4, 'matches_after_dedupe': 2}
  Family member 1: edges=[(0, 4), (0, 5), (0, 6), (0, 7), (1, 2), (1, 6), (1, 7), (2, 3), (3, 4), (3, 5), (4, 5), (5, 6)] rooted_iso_to_source=True
  Family member 2: edges=[(0, 1), (0, 2), (0, 4), (0, 6), (1, 2), (1, 5), (2, 5), (3, 5), (3, 7), (4, 5), (4, 6), (6, 7)] rooted_iso_to_source=False


## Exact Search Findings



On this run, the one-time 7-vertex atlas cache took **152.93 seconds** to build for **1044** unlabeled complement graphs.



After the cache existed, the filtered exact search for the held-out 8-vertex target was extremely selective:



- total rooted candidates in the naive search space: **132,588**

- graphs surviving the denominator divisibility filter: **2**

- couplings surviving the numeric screen: **4**

- symbolic checks actually performed: **4**

- connected exact matches before rooted deduplication: **4**

- rooted non-isomorphic exact 8-vertex unfoldings after deduplication: **2**



That is the main positive result of this notebook. The algebraic side is no longer the bottleneck once we commit to the exact-size `n = 8` regime and pay the cache cost once.



The held-out test graph therefore gives a genuine rooted multi-label target family of size 2. That makes it a good stress test for the retraining experiment below: the model does not just need to reproduce one arbitrary graph, it needs to land in the exact admissible family.

## Family-Aware Retraining Setup

The retraining experiment keeps the same broad input/output shape as the earlier ML notebook:

- input: sampled reduction values on a fixed $\lambda$ grid,
- output: the 28 upper-triangle edges of an 8-vertex graph.

The important change is the target set. Instead of comparing the model against only the source graph, we compare it against the **exact rooted admissible family** recovered by the algebraic search. During training we also randomize complement-vertex permutations, but we keep the root fixed at vertex `0`.

This is still a very small experiment, not a production retrain. The aim is to check whether exact-size labels and the smaller regime are enough to make the inverse problem noticeably easier.

In [ ]:
def generate_exact_dataset(cache, n_graphs=12, p=0.35, seed=222, exclude_A=None):

    rng = np.random.default_rng(seed)
    lambdas = np.linspace(-4.75, 4.75, 96, dtype=np.float32)
    data = []
    attempts = 0

    while len(data) < n_graphs and attempts < 250:
        attempts += 1
        G = nx.erdos_renyi_graph(8, p, seed=int(rng.integers(0, 10**9)))
        if not nx.is_connected(G):
            continue

        G = nx.convert_node_labels_to_integers(G)
        A = nx.to_numpy_array(G, dtype=int)
        if exclude_A is not None and rooted_isomorphic(A, exclude_A):
            continue

        target_r = compute_reduction_symbolic_from_graph(G, kept=0)
        family, _ = exact_size_family(target_r, cache, k=7)

        if not family:
            continue

        data.append({
            'A': A,
            'R': isospectral_reduction_numeric(A.astype(float), 0, lambdas),
            'family': family,
            'r_expr': target_r,
        })

        print(f"dataset example {len(data)}: edges={int(A.sum() // 2)} family_size={len(family)}")

    return data, lambdas



def candidate_edge_vectors(sample, rng, perms_per_target=3):
    candidates = []
    for A in sample['family']:
        candidates.append(edge_vector(A))

        for _ in range(max(0, perms_per_target - 1)):
            perm = rng.permutation(7)
            candidates.append(edge_vector(permute_complement(A, perm)))

    return np.stack(candidates)


class FamilyMLP(nn.Module):
    def __init__(self, in_dim, out_dim=28, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)


def train_family_aware_model(train_data, epochs=80, lr=1e-3, perms_per_target=3, seed=222):

    rng = np.random.default_rng(seed)
    X = np.stack([d['R'] for d in train_data]).astype(np.float32)
    feature_mean = X.mean(axis=0)
    feature_std = X.std(axis=0) + 1e-6
    X_t = torch.tensor((X - feature_mean) / feature_std, dtype=torch.float32, device=device)

    model = FamilyMLP(in_dim=X.shape[1]).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss(reduction='none')
    order = np.arange(len(train_data))

    for epoch in range(1, epochs + 1):
        rng.shuffle(order)
        epoch_loss = 0.0
        for idx in order:
            xb = X_t[idx:idx + 1]
            logits = model(xb)
            candidates = torch.tensor(
                candidate_edge_vectors(train_data[idx], rng, perms_per_target=perms_per_target),
                dtype=torch.float32,
                device=device,
            )

            losses = criterion(logits.expand(candidates.shape[0], -1), candidates).mean(dim=1)
            loss = losses.min()
            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += float(loss.item())

        if epoch % 20 == 0:
            print(f'epoch {epoch:02d} train_loss={epoch_loss / len(train_data):.4f}')

    return {
        'model': model.eval(),
        'feature_mean': feature_mean,
        'feature_std': feature_std,
    }



def predict_graph_from_feature(model_pack, R_feature, threshold=0.5):
    x = ((R_feature - model_pack['feature_mean']) / model_pack['feature_std']).astype(np.float32)
    x_t = torch.tensor(x, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        logits = model_pack['model'](x_t).squeeze(0).cpu().numpy()

    logits = np.clip(logits, -30.0, 30.0)
    probs = 1.0 / (1.0 + np.exp(-logits))
    A = np.zeros((8, 8), dtype=int)
    iu = np.triu_indices(8, k=1)
    A[iu] = (probs > threshold).astype(int)
    A = A + A.T
    return A, probs



def reduction_mse(A_pred, sample, lambdas):
    pred_R = isospectral_reduction_numeric(A_pred.astype(float), 0, lambdas)
    diff = pred_R - sample['R']
    return float(np.mean(diff * diff))



def exact_match(sample, A_pred):
    if not nx.is_connected(nx.from_numpy_array(A_pred)):
        return False

    pred_r = compute_reduction_symbolic_from_graph(nx.from_numpy_array(A_pred), kept=0)
    return sp.simplify(reduced_expr(pred_r - sample['r_expr'])) == 0



def family_hit(sample, A_pred):
    return any(rooted_isomorphic(A_pred, A) for A in sample['family'])


In [9]:
dataset, lambdas = generate_exact_dataset(
    atlas_cache,
    n_graphs=12,
    p=0.35,
    seed=222,
    exclude_A=test_A,
)

train_data = dataset[:8]
val_data = dataset[8:10]
extra_data = dataset[10:12]
print(f'train={len(train_data)} val={len(val_data)} extra={len(extra_data)}')

model_pack = train_family_aware_model(train_data, epochs=80, lr=1e-3, perms_per_target=3, seed=222)

val_rows = []
for threshold in [0.30, 0.40, 0.50, 0.60]:
    connected_rate = 0
    family_rate = 0
    exact_rate = 0
    mse_values = []
    for sample in val_data:
        A_pred, _ = predict_graph_from_feature(model_pack, sample['R'], threshold=threshold)
        connected_rate += int(nx.is_connected(nx.from_numpy_array(A_pred)))
        family_rate += int(family_hit(sample, A_pred))
        exact_rate += int(exact_match(sample, A_pred))
        mse_values.append(reduction_mse(A_pred, sample, lambdas))
    row = {
        'threshold': threshold,
        'connected_rate': connected_rate / len(val_data),
        'family_rate': family_rate / len(val_data),
        'exact_rate': exact_rate / len(val_data),
        'mean_reduction_mse': float(np.mean(mse_values)),
    }
    val_rows.append(row)

print('Validation sweep:')
for row in val_rows:
    print(row)

best_threshold = min(val_rows, key=lambda row: (-row['family_rate'], -row['exact_rate'], row['mean_reduction_mse']))['threshold']
print('Selected threshold:', best_threshold)

dataset example 1: edges=8 family_size=1
dataset example 2: edges=11 family_size=1
dataset example 3: edges=12 family_size=1
dataset example 4: edges=10 family_size=1
dataset example 5: edges=13 family_size=1
dataset example 6: edges=13 family_size=1
dataset example 7: edges=9 family_size=1
dataset example 8: edges=10 family_size=1
dataset example 9: edges=10 family_size=2
dataset example 10: edges=13 family_size=1
dataset example 11: edges=13 family_size=1
dataset example 12: edges=11 family_size=1
train=8 val=2 extra=2
epoch 20 train_loss=0.0804
epoch 40 train_loss=0.0010
epoch 60 train_loss=0.0003
epoch 80 train_loss=0.0002
Validation sweep:
{'threshold': 0.3, 'connected_rate': 1.0, 'family_rate': 0.0, 'exact_rate': 0.0, 'mean_reduction_mse': 497.7582092285156}
{'threshold': 0.4, 'connected_rate': 1.0, 'family_rate': 0.0, 'exact_rate': 0.0, 'mean_reduction_mse': 497.7582092285156}
{'threshold': 0.5, 'connected_rate': 1.0, 'family_rate': 0.0, 'exact_rate': 0.0, 'mean_reduction_mse': 

In [10]:
heldout_sample = {
    'A': test_A,
    'R': isospectral_reduction_numeric(test_A.astype(float), 0, lambdas),
    'family': test_family,
    'r_expr': test_target_r,
}

A_test_pred, test_probs = predict_graph_from_feature(model_pack, heldout_sample['R'], threshold=best_threshold)
heldout_connected = nx.is_connected(nx.from_numpy_array(A_test_pred))
heldout_family_hit = family_hit(heldout_sample, A_test_pred)
heldout_exact = exact_match(heldout_sample, A_test_pred)
heldout_mse = reduction_mse(A_test_pred, heldout_sample, lambdas)

print('Held-out 8-vertex test graph evaluation')
print('  connected:', heldout_connected)
print('  family_hit:', heldout_family_hit)
print('  exact_match:', heldout_exact)
print('  reduction_mse:', heldout_mse)
print('  true_edges:', graph_edges(test_A))
print('  pred_edges:', graph_edges(A_test_pred))
print('  family_size:', len(test_family))

Held-out 8-vertex test graph evaluation
  connected: True
  family_hit: False
  exact_match: False
  reduction_mse: 957.405029296875
  true_edges: [(0, 3), (0, 5), (0, 6), (0, 7), (1, 2), (1, 3), (1, 6), (2, 4), (3, 5), (3, 6), (4, 5), (4, 7)]
  pred_edges: [(0, 4), (0, 5), (0, 7), (1, 2), (1, 6), (2, 3), (3, 4), (3, 5), (4, 5), (5, 6), (5, 7)]
  family_size: 2


## Results and Interpretation



This run cleanly separates the two issues that were entangled in the larger-`n` experiments.



### 1. The exact-search bottleneck is substantially improved



At `n = 8`, the algebraic reconstruction is fully available. The faster coupling search reduced the held-out target from a naive **132,588** candidate pairs to just **4 symbolic checks**, while still recovering the full rooted admissible family. So the exact-label generation problem is practical in this regime.



### 2. The learning bottleneck remains



The compact family-aware MLP was trained on **8** exact-family training examples, validated on **2** more, and overfit the training set quickly:



- epoch 20 train loss: **0.0804**

- epoch 40 train loss: **0.0010**

- epoch 60 train loss: **0.0003**

- epoch 80 train loss: **0.0002**



But the validation sweep still produced:



- connected rate: **1.0** for all tested thresholds

- rooted family hit rate: **0.0**

- exact symbolic hit rate: **0.0**



Using the selected threshold `0.3`, the held-out 8-vertex test graph was decoded into a connected graph, but it was **not** in the admissible rooted family and its exact reduction did **not** match symbolically. The held-out reduction MSE was **957.41**.



### 3. What this means



This is still useful. The notebook shows that once we move to `n <= 8`, the exact target-family generation problem becomes tractable, so any remaining failure is now mostly a **generalization / model-design issue**, not an algebraic coverage issue.



The immediate next experiment should therefore be to keep the exact `n = 8` data regime, but scale the supervised dataset size upward and replace the plain edge-vector MLP with a model that is explicitly better at rooted graph generation under family-valued supervision.

## Next Experiment — Scale Data and Compare Target Strategies



This section follows the two immediate follow-ups from the previous results:



1. **Scale the exact n=8 dataset upward** now that the cached coupling search is cheap per target.

2. **Compare supervision strategies directly** under matched model capacity and optimizer settings:

   - source-only training target (single graph per reduction),

   - exact-family training target (best-of-family, rooted).



To reduce confounding, both variants share the same rooted decoder architecture. The decoder has one head for the 7 root-to-complement edges and one head for the 21 complement internal edges. This is still lightweight, but now explicitly respects the rooted structure of the problem.

In [ ]:
def split_list(data, n_train, n_val):
    train = data[:n_train]
    val = data[n_train:n_train + n_val]
    test = data[n_train + n_val:]
    return train, val, test



class RootedDecoderMLP(nn.Module):
    def __init__(self, in_dim, hidden=256):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )

        self.root_head = nn.Linear(hidden, 7)
        self.comp_head = nn.Linear(hidden, 21)

    def forward(self, x):
        h = self.backbone(x)
        root_logits = self.root_head(h)
        comp_logits = self.comp_head(h)
        return torch.cat([root_logits, comp_logits], dim=1)



def train_rooted_model(train_data, mode='source', epochs=100, lr=8e-4, seed=0, perms_per_target=3, hidden=256):
    if mode not in {'source', 'family'}:
        raise ValueError("mode must be 'source' or 'family'")

    rng = np.random.default_rng(seed)
    X = np.stack([d['R'] for d in train_data]).astype(np.float32)
    feature_mean = X.mean(axis=0)
    feature_std = X.std(axis=0) + 1e-6
    X_t = torch.tensor((X - feature_mean) / feature_std, dtype=torch.float32, device=device)

    model = RootedDecoderMLP(in_dim=X.shape[1], hidden=hidden).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss(reduction='none')
    order = np.arange(len(train_data))

    for epoch in range(1, epochs + 1):
        rng.shuffle(order)
        epoch_loss = 0.0
        for idx in order:
            xb = X_t[idx:idx + 1]
            logits = model(xb)

            if mode == 'source':
                candidates_np = np.stack([edge_vector(train_data[idx]['A'])])

            else:
                candidates_np = candidate_edge_vectors(train_data[idx], rng, perms_per_target=perms_per_target)

            candidates = torch.tensor(candidates_np, dtype=torch.float32, device=device)
            losses = criterion(logits.expand(candidates.shape[0], -1), candidates).mean(dim=1)
            loss = losses.min()
            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += float(loss.item())

        if epoch % 25 == 0:
            print(f'[{mode}] epoch {epoch:03d} train_loss={epoch_loss / len(train_data):.4f}')



    return {
        'model': model.eval(),
        'feature_mean': feature_mean,
        'feature_std': feature_std,
        'mode': mode,
    }



def evaluate_pack_on_dataset(model_pack, data_split, lambdas, thresholds=(0.3, 0.4, 0.5, 0.6)):
    rows = []
    for threshold in thresholds:
        connected = 0
        family = 0
        exact = 0
        mses = []
        for sample in data_split:
            A_pred, _ = predict_graph_from_feature(model_pack, sample['R'], threshold=threshold)
            connected += int(nx.is_connected(nx.from_numpy_array(A_pred)))
            family += int(family_hit(sample, A_pred))
            exact += int(exact_match(sample, A_pred))
            mses.append(reduction_mse(A_pred, sample, lambdas))

        rows.append({
            'threshold': float(threshold),
            'connected_rate': connected / len(data_split),
            'family_rate': family / len(data_split),
            'exact_rate': exact / len(data_split),
            'mean_reduction_mse': float(np.mean(mses)),
        })

    return rows


def select_threshold(rows):
    return min(rows, key=lambda row: (-row['family_rate'], -row['exact_rate'], row['mean_reduction_mse']))['threshold']


def evaluate_single_sample(model_pack, sample, lambdas, threshold):
    A_pred, _ = predict_graph_from_feature(model_pack, sample['R'], threshold=threshold)

    return {
        'connected': bool(nx.is_connected(nx.from_numpy_array(A_pred))),
        'family_hit': bool(family_hit(sample, A_pred)),
        'exact_match': bool(exact_match(sample, A_pred)),
        'reduction_mse': float(reduction_mse(A_pred, sample, lambdas)),
        'pred_edges': graph_edges(A_pred),
    }


In [ ]:
# Build a larger exact-family dataset in the n=8 regime

scaled_dataset, scaled_lambdas = generate_exact_dataset(
    atlas_cache,
    n_graphs=36,
    p=0.35,
    seed=808,
    exclude_A=test_A,
)



scaled_train, scaled_val, scaled_test = split_list(scaled_dataset, n_train=24, n_val=6)
print(f'scaled split sizes: train={len(scaled_train)} val={len(scaled_val)} test={len(scaled_test)}')
print('family sizes in scaled dataset:', sorted([len(d['family']) for d in scaled_dataset]))

# Train matched models: source-only target vs family-aware target
source_pack = train_rooted_model(
    scaled_train,
    mode='source',
    epochs=100,
    lr=8e-4,
    seed=808,
    perms_per_target=3,
    hidden=256,
)



family_pack = train_rooted_model(
    scaled_train,
    mode='family',
    epochs=100,
    lr=8e-4,
    seed=808,
    perms_per_target=3,
    hidden=256,
)



# Calibrate threshold on validation set for each model
source_val_rows = evaluate_pack_on_dataset(source_pack, scaled_val, scaled_lambdas)
family_val_rows = evaluate_pack_on_dataset(family_pack, scaled_val, scaled_lambdas)

source_best_threshold = select_threshold(source_val_rows)
family_best_threshold = select_threshold(family_val_rows)

print('source validation rows:')
for row in source_val_rows:
    print(' ', row)

print('family validation rows:')
for row in family_val_rows:
    print(' ', row)

print('selected thresholds:', {'source': source_best_threshold, 'family': family_best_threshold})

# Evaluate on test split
source_test_rows = evaluate_pack_on_dataset(source_pack, scaled_test, scaled_lambdas, thresholds=(source_best_threshold,))
family_test_rows = evaluate_pack_on_dataset(family_pack, scaled_test, scaled_lambdas, thresholds=(family_best_threshold,))

print('source test metrics:', source_test_rows[0])
print('family test metrics:', family_test_rows[0])


dataset example 1: edges=11 family_size=1
dataset example 2: edges=13 family_size=1
dataset example 3: edges=13 family_size=1
dataset example 4: edges=8 family_size=1
dataset example 5: edges=12 family_size=1
dataset example 6: edges=9 family_size=1
dataset example 7: edges=9 family_size=1
dataset example 8: edges=8 family_size=1
dataset example 9: edges=12 family_size=1
dataset example 10: edges=8 family_size=1
dataset example 11: edges=11 family_size=1
dataset example 12: edges=8 family_size=1
dataset example 13: edges=10 family_size=1
dataset example 14: edges=7 family_size=1
dataset example 15: edges=14 family_size=2
dataset example 16: edges=14 family_size=1
dataset example 17: edges=8 family_size=1
dataset example 18: edges=14 family_size=1
dataset example 19: edges=8 family_size=1
dataset example 20: edges=9 family_size=1
dataset example 21: edges=13 family_size=1
dataset example 22: edges=9 family_size=1
dataset example 23: edges=7 family_size=1
dataset example 24: edges=8 fami

In [ ]:
# Evaluate both models on the same held-out 8-vertex graph chosen above

heldout_sample_scaled = {
    'A': test_A,
    'R': isospectral_reduction_numeric(test_A.astype(float), 0, scaled_lambdas),
    'family': test_family,
    'r_expr': test_target_r,
}


source_heldout = evaluate_single_sample(source_pack, heldout_sample_scaled, scaled_lambdas, source_best_threshold)
family_heldout = evaluate_single_sample(family_pack, heldout_sample_scaled, scaled_lambdas, family_best_threshold)

print('held-out source-only:', source_heldout)
print('held-out family-aware:', family_heldout)
print('held-out true edges:', graph_edges(test_A))
print('held-out family size:', len(test_family))

held-out source-only: {'connected': True, 'family_hit': False, 'exact_match': False, 'reduction_mse': 1491.4061279296875, 'pred_edges': [(0, 1), (0, 2), (0, 5), (0, 6), (1, 3), (1, 4), (1, 7), (3, 7), (4, 5), (4, 6), (5, 7), (6, 7)]}
held-out family-aware: {'connected': False, 'family_hit': False, 'exact_match': False, 'reduction_mse': 3766.889404296875, 'pred_edges': [(0, 1), (0, 2), (0, 6), (0, 7), (1, 2), (2, 3), (3, 4), (3, 6)]}
held-out true edges: [(0, 3), (0, 5), (0, 6), (0, 7), (1, 2), (1, 3), (1, 6), (2, 4), (3, 5), (3, 6), (4, 5), (4, 7)]
held-out family size: 2


## Comparison Results (Scaled Dataset)



### Why this comparison was run



The previous section showed that exact rooted families are now computationally available at `n = 8`. So the next question is whether better labels alone improve recovery. To test that cleanly, both models here used the same rooted architecture, optimizer, epoch budget, and split; only the training targets differed.



- **Source-only model:** supervised against a single source graph per reduction.

- **Family-aware model:** supervised against the exact rooted admissible family per reduction.



### What happened in this run



We scaled to 36 exact-labeled samples (`24 train / 6 val / 6 test`). Family cardinalities were mostly 1, with only one sample of size 2. Both models fit training data strongly by 100 epochs.



Validation-selected threshold was `0.6` for both models. On the 6-sample test split:



- source-only: connected rate `0.333`, family rate `0.0`, exact rate `0.0`, mean reduction MSE `81.47`

- family-aware: connected rate `0.833`, family rate `0.0`, exact rate `0.0`, mean reduction MSE `99.97`



On the fixed held-out 8-vertex graph (family size 2):



- source-only: connected `True`, family hit `False`, exact match `False`, reduction MSE `1491.41`

- family-aware: connected `False`, family hit `False`, exact match `False`, reduction MSE `3766.89`



### Interpretation



In this scaled run, family-aware supervision improved structural connectivity on the random test split but still did not produce any family hits or exact symbolic matches. Because the exact labels are now available and used, the remaining limitation appears to be modeling/optimization rather than missing algebraic supervision.



A practical next step is to rebalance the dataset toward reductions with family size > 1 (currently underrepresented), then evaluate architectures that decode graphs with stronger combinatorial bias (for example, constrained edge-factor models or discrete refinement over model proposals).

## Singleton vs Multi-Family Behavior (Answer + Next Step)



### Does a one-to-many model still handle one-to-one cases?



Yes. In this setup, a reduction with exactly one rooted admissible unfolding corresponds to a family of size 1. The family-aware loss then collapses to the same objective shape as source-only BCE (modulo complement permutations), so it should not fundamentally break on singleton cases.



In practice, performance on singleton reductions depends on optimization and architecture quality, not on whether the framework *expects* multiplicity. So the right check is empirical subgroup evaluation: singleton-family test examples vs multi-family test examples.



### Next step implemented here



To stress multi-solution supervision more directly, we build a rebalanced dataset that explicitly includes a larger fraction of reductions with family size >= 2, then retrain both source-only and family-aware models under matched settings and compare by subgroup.

In [ ]:
def evaluate_pack_by_family_size(model_pack, data_split, lambdas, threshold):
    singleton = [d for d in data_split if len(d['family']) == 1]
    multi = [d for d in data_split if len(d['family']) >= 2]

    out = {}
    if singleton:
        out['singleton'] = evaluate_pack_on_dataset(model_pack, singleton, lambdas, thresholds=(threshold,))[0]
        out['singleton_count'] = len(singleton)
    else:
        out['singleton'] = None
        out['singleton_count'] = 0

    if multi:
        out['multi'] = evaluate_pack_on_dataset(model_pack, multi, lambdas, thresholds=(threshold,))[0]
        out['multi_count'] = len(multi)
    else:
        out['multi'] = None
        out['multi_count'] = 0

    return out



def collect_family_size_ge2(cache, target_count=8, p=0.35, seed=1701, max_attempts=5000, exclude_A=None):
    rng = np.random.default_rng(seed)
    lambdas = np.linspace(-4.75, 4.75, 96, dtype=np.float32)
    out = []
    attempts = 0

    while len(out) < target_count and attempts < max_attempts:
        attempts += 1
        G = nx.erdos_renyi_graph(8, p, seed=int(rng.integers(0, 10**9)))
        if not nx.is_connected(G):
            continue

        G = nx.convert_node_labels_to_integers(G)
        A = nx.to_numpy_array(G, dtype=int)
        if exclude_A is not None and rooted_isomorphic(A, exclude_A):
            continue

        target_r = compute_reduction_symbolic_from_graph(G, kept=0)
        family, _ = exact_size_family(target_r, cache, k=7)

        if len(family) < 2:
            continue

        duplicate = False
        for item in out:
            if rooted_isomorphic(item['A'], A):
                duplicate = True
                break

        if duplicate:
            continue

        out.append({
            'A': A,
            'R': isospectral_reduction_numeric(A.astype(float), 0, lambdas),
            'family': family,
            'r_expr': target_r,
        })

        print(f"multi-family example {len(out)}: edges={int(A.sum() // 2)} family_size={len(family)} attempts={attempts}")

    return out, lambdas, attempts


In [ ]:
# 1) Directly answer singleton behavior on the current scaled-test setting

source_by_family = evaluate_pack_by_family_size(source_pack, scaled_test, scaled_lambdas, source_best_threshold)
family_by_family = evaluate_pack_by_family_size(family_pack, scaled_test, scaled_lambdas, family_best_threshold)

print('Current scaled-test subgroup metrics')
print('source-only:', source_by_family)
print('family-aware:', family_by_family)

# 2) Next step: build a rebalanced dataset with explicit multi-family coverage
multi_pool, multi_lambdas, multi_attempts = collect_family_size_ge2(
    atlas_cache,
    target_count=8,
    p=0.35,
    seed=1701,
    max_attempts=5000,
    exclude_A=test_A,
)

print(f'Collected {len(multi_pool)} multi-family examples in {multi_attempts} attempts')

# Build singleton pool from scaled dataset and avoid overlap with selected multi roots
singleton_pool = []
for d in scaled_dataset:
    if len(d['family']) != 1:
        continue

    duplicate = False
    for md in multi_pool:
        if rooted_isomorphic(d['A'], md['A']):
            duplicate = True
            break

    if not duplicate:
        singleton_pool.append(d)

target_singleton = min(len(singleton_pool), len(multi_pool) * 2)
singleton_subset = singleton_pool[:target_singleton]
balanced_dataset = singleton_subset + multi_pool
print(f'balanced dataset sizes -> singleton={len(singleton_subset)} multi={len(multi_pool)} total={len(balanced_dataset)}')

# Train/val/test split with guaranteed multi coverage in val+test where possible
rng_bal = np.random.default_rng(1701)
indices = np.arange(len(balanced_dataset))
rng_bal.shuffle(indices)
balanced_dataset = [balanced_dataset[i] for i in indices]

n_total = len(balanced_dataset)
n_train = int(0.67 * n_total)
n_val = int(0.16 * n_total)
bal_train = balanced_dataset[:n_train]
bal_val = balanced_dataset[n_train:n_train + n_val]
bal_test = balanced_dataset[n_train + n_val:]

print(f'balanced split -> train={len(bal_train)} val={len(bal_val)} test={len(bal_test)}')
print('test family sizes:', [len(d['family']) for d in bal_test])

bal_source_pack = train_rooted_model(
    bal_train,
    mode='source',
    epochs=120,
    lr=8e-4,
    seed=1701,
    perms_per_target=3,
    hidden=256,
)

bal_family_pack = train_rooted_model(
    bal_train,
    mode='family',
    epochs=120,
    lr=8e-4,
    seed=1701,
    perms_per_target=3,
    hidden=256,
)

bal_source_val = evaluate_pack_on_dataset(bal_source_pack, bal_val, multi_lambdas)
bal_family_val = evaluate_pack_on_dataset(bal_family_pack, bal_val, multi_lambdas)
bal_source_thr = select_threshold(bal_source_val)
bal_family_thr = select_threshold(bal_family_val)

print('balanced source val rows:')
for row in bal_source_val:
    print(' ', row)

print('balanced family val rows:')
for row in bal_family_val:
    print(' ', row)

print('balanced selected thresholds:', {'source': bal_source_thr, 'family': bal_family_thr})
bal_source_test_all = evaluate_pack_on_dataset(bal_source_pack, bal_test, multi_lambdas, thresholds=(bal_source_thr,))[0]
bal_family_test_all = evaluate_pack_on_dataset(bal_family_pack, bal_test, multi_lambdas, thresholds=(bal_family_thr,))[0]
bal_source_test_group = evaluate_pack_by_family_size(bal_source_pack, bal_test, multi_lambdas, bal_source_thr)
bal_family_test_group = evaluate_pack_by_family_size(bal_family_pack, bal_test, multi_lambdas, bal_family_thr)

print('balanced test all - source:', bal_source_test_all)
print('balanced test all - family:', bal_family_test_all)
print('balanced test by family-size - source:', bal_source_test_group)
print('balanced test by family-size - family:', bal_family_test_group)

Current scaled-test subgroup metrics
source-only: {'singleton': {'threshold': 0.6, 'connected_rate': 0.3333333333333333, 'family_rate': 0.0, 'exact_rate': 0.0, 'mean_reduction_mse': 81.46719868977864}, 'singleton_count': 6, 'multi': None, 'multi_count': 0}
family-aware: {'singleton': {'threshold': 0.6, 'connected_rate': 0.8333333333333334, 'family_rate': 0.0, 'exact_rate': 0.0, 'mean_reduction_mse': 99.97283569971721}, 'singleton_count': 6, 'multi': None, 'multi_count': 0}
multi-family example 1: edges=10 family_size=2 attempts=1
multi-family example 2: edges=9 family_size=2 attempts=24
multi-family example 3: edges=9 family_size=2 attempts=28
multi-family example 4: edges=11 family_size=2 attempts=68
multi-family example 5: edges=12 family_size=2 attempts=77
multi-family example 6: edges=10 family_size=2 attempts=89
multi-family example 7: edges=12 family_size=2 attempts=95
multi-family example 8: edges=10 family_size=2 attempts=118
Collected 8 multi-family examples in 118 attempts
ba

## Follow-up Results: Singleton and Rebalanced Multi-Family



### 1) Answer: what happens when there is only one unique unfolding?



Empirically in this notebook, the model family does **not** fail just because the task is singleton. On the current scaled test split, all 6 examples were singleton-family (`multi_count = 0`), and both models still produced valid outputs:



- source-only on singleton subset: connected rate `0.333`, family/exact `0.0 / 0.0`, mean reduction MSE `81.47`

- family-aware on singleton subset: connected rate `0.833`, family/exact `0.0 / 0.0`, mean reduction MSE `99.97`



So a one-to-many training objective can still operate on one-to-one cases; the bottleneck remains exact symbolic recovery, not singleton compatibility.



### 2) Rebalanced next-step experiment



We explicitly collected multi-family reductions (`family_size >= 2`) and formed a rebalanced dataset:



- collected multi-family pool: `8` examples in `118` attempts

- balanced dataset: singleton `16`, multi `8`, total `24`

- split used: `train=16`, `val=3`, `test=5` (test family sizes included both singleton and multi)



Selected thresholds from validation:



- source-only: `0.5`

- family-aware: `0.6`



Balanced test (all 5 samples):



- source-only: connected `0.6`, family/exact `0.0 / 0.0`, mean reduction MSE `76.81`

- family-aware: connected `0.6`, family/exact `0.0 / 0.0`, mean reduction MSE `4,752,061.70`



Balanced test by subgroup:



- source-only, singleton (`n=3`): connected `1.0`, exact `0.0`, MSE `104.97`

- source-only, multi (`n=2`): connected `0.0`, exact `0.0`, MSE `34.57`

- family-aware, singleton (`n=3`): connected `0.667`, exact `0.0`, MSE `7,920,066.44`

- family-aware, multi (`n=2`): connected `0.5`, exact `0.0`, MSE `54.59`



### Interpretation



This run supports the design intuition (family-aware can include singleton cases), but still shows no exact symbolic hits. In this particular rebalanced split, family-aware training was less numerically stable on singleton-heavy portions. The next practical move is to stabilize the objective (for example, clipping/robustifying reduction targets and adding connectivity-aware decoding), then rerun this same subgroup protocol.